## Fixed-Length Sliding Windows

### Maximum Sum of Subarrays

Given an array of integers nums and an integer k, find the maximum sum of any contiguous subarray of size k.

Ex.
```python
nums = [2, 1, 5, 1, 3, 2]
k = 3
```

First up we have a "naive" approach using listcomps and a helper function for readability:

In [77]:
def maximum_sum_of_subarrays(nums: list[int], k: int) -> int:
    def windows(nums: list[int], k: int) -> list[list[int]]:
        return [nums[i:i+k] for i in range(len(nums) - k + 1)]

    sums = [sum(w) for w in windows(nums, k)]
    return max(sums)

nums = [2, 1, 5, 1, 3, 2]
k = 3
print(maximum_sum_of_subarrays(nums, k))

9


One thing to note is that the number of subarrays of length k in a list `nums` of length n is always going to equal `len(nums) - k + 1`.
Once you know that the solution above reads nicely but it's suboptimal. We could express it even more concisely:
```python
return max(sum(w) for w in (nums[i:i+k] for i in range(len(nums) - k + 1)))
```
But the problem is `sum(w)` rescans each window. So, it's `O(n*k)`. But look how closely it mirrors the problem statement:
```
find the maximum sum of any contiguous subarray of size k
max(sum(w) for w in windows(nums, k))
```
Alright, the solution to the compute complexity is using *prefix sums*.

In [70]:
from itertools import accumulate

def maximum_sum_of_subarrays_v2(nums: list[int], k: int) -> int:
    """Use prefix sums
    """
    prefix = [0] + list(accumulate(nums))
    # [2, 3, 8, 9, 12, 14]
    # accumulate is a scan (similar to Haskell's scanl)
    # that
    sums = [prefix[i+k] - prefix[i] for i in range(len(nums) - k + 1)]
    return max(sums)

In [71]:
nums = [2, 1, 5, 1, 3, 2]
k = 3
maximum_sum_of_subarrays_v2(nums, k)

9

### Max Sum of Distinct Subarrays

OK now let's do one with a condition - the subarray cannot include any duplicate values.

In [75]:
def maximum_unique_sum_of_subarrays(nums: list[int], k: int) -> int:
    def windows(nums: list[int], k: int) -> list[list[int]]:
        return (nums[i:i+k] for i in range(len(nums) - k + 1))

    def is_unique(window: list[int]) -> bool:
        return len(set(window)) == len(window)

    return max(sum(w) for w in windows(nums, k) if is_unique(w))

In [76]:
nums = [3, 2, 2, 3, 4, 6, 7, 7, -1]
k = 4
maximum_unique_sum_of_subarrays(nums, k)

20

Now let's do this one:
```
Given an array of integers representing card values, write a function to calculate the maximum score you can achieve by picking exactly k cards.

You must pick cards in order from either end. You can take some cards from the beginning, then switch to taking cards from the end, but you cannot skip cards or pick from the middle.
```

## Variable-Length Sliding Windows

### Longest Substring

In [83]:
def longest_substring(s: str) -> int:

    def is_valid(state) -> bool:
        return max(state.values()) <= 1

    def shrink_until_valid(s: str, freq_dict: dict, start: int):
        while not is_valid(freq_dict):
            freq_dict[s[start]] -= 1
            if freq_dict[s[start]] == 0:
                del freq_dict[start]
            start += 1
        return freq_dict, start

    def window_length(s: str):
        freq_dict, start = {}, 0
        for end, ch in enumerate(s):
            freq_dict[ch] = freq_dict.get(ch, 0) + 1
            freq_dict, start = shrink_until_valid(s, freq_dict, start)
            yield end - start + 1

    return max(window_length(s), default=0)

In [84]:
s = "substring"
print(longest_substring(s))

8


### Longest Repeating Character Substring

In [87]:
def longest_repeating(s: str) -> int:
    pass

## Intervals

### Can Attend Meetings

In [15]:
def can_attend(intervals: list[int, int]) -> bool:
    meetings = sorted(intervals)
    return all(a[1] <= b[0] for a, b in zip(meetings, meetings[1:]))

In [16]:
meetings = [(1, 3), (4, 6), (5, 7), (7, 10), (11, 12)]
can_attend(meetings)


False

#### Interlude: Python's `reduce()`

In [ ]:
from functools import reduce

nums = [1, 2, 3, 4 ,5]

# Classics for reference
sum_ = reduce(lambda acc, x: acc + x, nums)
product = reduce(lambda acc, x: acc * x, nums)
maximum = reduce(lambda acc, x: max(acc, x), nums)
minimum = reduce(lambda acc, x: min(acc, x), nums)

# Flattening, actually useful
nested = [[1, 2], [3, 4], [5, 6]]
flatten = reduce(lambda acc, x: acc + x, nested)

# Counting occurrences
words  = ["a", "b", "a", "c", "b", "a"]

# Merging intervals

# Pipeline of functions


### Insert Interval

In [25]:
from functools import reduce

def insert_interval(intervals: list[tuple[int, int]], interval: tuple[int, int]) -> list[tuple[int, int]]:
    meetings = sorted(intervals)
    before = [m for m in meetings if m[1] <= interval[0]]
    overlap = [m for m in meetings if m[0] <= interval[1] and m[1] > interval[0]]
    after = [m for m in meetings if m[0] > interval[1]]

    merged = reduce(
        lambda acc, iv: [min(acc[0], iv[0]), max(acc[1], iv[1])],
        overlap,
        interval
    )

    return before + [merged] + after

In [26]:
intervals = [[1,3],[6,9]]
newInterval = [2,5]
insert_interval(intervals, newInterval)

[[1, 5], [6, 9]]

In [49]:
def permutations(nums: list[int]):
    if not nums:
        yield []
        return
    for i, x in enumerate(nums):
        for rest in permutations(nums[:i] + nums[i+1:]):
            yield [x] + rest

In [50]:
list(permutations([1,2,3]))

[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]